**V1 experiment.** This notebook contains the V1 implementation and experiment configuration. See [methods and evaluation](../docs/METHODS.md) for the implemented protocol.


# Fix 2 — ResNet-18 baselines on consistent 50-epoch settings (Levels 1, 2)

All baselines:
  - Same architecture (`utils.cnn.make_resnet18`).
  - Same per-spectrogram normalization (`utils.cnn._normalize_specs`).
  - 50 epochs of Adam at lr=1e-3, batch=32, seed=0.
  - Train on `(is_train & labeled_clip_mask)`; evaluate on `~is_train`.

Configurations:

| Level | Run name        | Labeled subset                  |
|---|---|---|
| 1 | L1-fair-50      | clip-level 50% (trial 0)        |
| 1 | L1-full         | 100% of HF train (no SSL mask)  |
| 2 | L2-fair-30      | clip-level 30% (trial 0)        |
| 2 | L2-fair-50      | clip-level 50% (trial 0)        |
| 2 | L2-fair-75      | clip-level 75% (trial 0)        |
| 2 | L2-full         | 100% of HF train                |

Per-epoch logs (train_loss, train_acc, test_acc) and the final test
accuracy / macro F1 / confusion matrix are written into
`v2/Level{1,2}/figures/` and `v2/Level{1,2}/resnet_v2_results.json`.


In [ ]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "utils" / "graph.py").is_file() and
           (PROJECT_ROOT / "v2" / "utils" / "splits.py").is_file()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Start Jupyter inside the cloned repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.cnn import make_resnet18, _normalize_specs
from utils.evaluation import plot_confusion_matrix
from v2.utils.runs import make_run_dir

V2_ROOT = PROJECT_ROOT / "v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Per-run output folders: this notebook touches both Levels 1 and 2, so
# create a fresh run inside each. Edit the notebook between executions
# to compare configurations.
L1_RUN = make_run_dir(V2_ROOT / "Level1", notebook_label="resnet18_v2.ipynb")
L2_RUN = make_run_dir(V2_ROOT / "Level2", notebook_label="resnet18_v2.ipynb")
print(f"Level 1 run -> {L1_RUN}")
print(f"Level 2 run -> {L2_RUN}")

EPOCHS = 50
BATCH = 32
LR = 1e-3
SEED = 0


## Training helper (logs per-epoch loss/train_acc/test_acc)


In [ ]:
def train_one(
    name: str,
    S_train, y_train, S_test, y_test,
    num_classes: int,
    epochs: int = EPOCHS,
    batch_size: int = BATCH,
    lr: float = LR,
    seed: int = SEED,
):
    torch.manual_seed(seed); np.random.seed(seed)
    Xtr = _normalize_specs(S_train)[:, None, :, :]
    Xte = _normalize_specs(S_test)[:, None, :, :]
    Xtr_t = torch.from_numpy(Xtr); ytr_t = torch.from_numpy(y_train.astype(np.int64))
    Xte_t = torch.from_numpy(Xte); yte_t = torch.from_numpy(y_test.astype(np.int64))
    dl = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size,
                    shuffle=True, drop_last=False)
    model = make_resnet18(num_classes).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()

    train_losses, train_accs, test_accs = [], [], []

    @torch.no_grad()
    def eval_acc(X, y):
        model.eval()
        preds = []
        for i in range(0, X.shape[0], batch_size):
            xb = X[i:i+batch_size].to(device)
            preds.append(model(xb).argmax(dim=1).cpu().numpy())
        if not preds:
            return 0.0
        return float(accuracy_score(y.numpy(), np.concatenate(preds)))

    t0 = time.time()
    for ep in range(epochs):
        model.train()
        ep_loss, correct, total = 0.0, 0, 0
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            out = model(xb)
            loss = crit(out, yb)
            loss.backward()
            opt.step()
            ep_loss += loss.item() * xb.size(0)
            correct += (out.argmax(dim=1) == yb).sum().item()
            total += xb.size(0)
        train_losses.append(ep_loss / max(total, 1))
        train_accs.append(correct / max(total, 1))
        test_accs.append(eval_acc(Xte_t, yte_t))
    dur = time.time() - t0

    # Final predictions
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, Xte_t.shape[0], batch_size):
            xb = Xte_t[i:i+batch_size].to(device)
            preds.append(model(xb).argmax(dim=1).cpu().numpy())
    y_pred = np.concatenate(preds) if preds else np.zeros(0, dtype=np.int64)

    final_acc = float(accuracy_score(y_test, y_pred))
    final_f1 = float(f1_score(y_test, y_pred, average="macro",
                               labels=range(num_classes), zero_division=0))
    print(f"  [{name}] {epochs} ep done in {dur:.1f}s | "
          f"final test acc={final_acc:.4f}, F1={final_f1:.4f}")
    return {
        "name": name,
        "epochs": epochs,
        "train_loss": train_losses,
        "train_acc": train_accs,
        "test_acc": test_accs,
        "final_acc": final_acc,
        "final_f1": final_f1,
        "y_pred": y_pred,
        "y_test": y_test,
        "n_train": int(S_train.shape[0]),
        "n_test": int(S_test.shape[0]),
        "wall_seconds": float(dur),
    }


def save_run_artifacts(run, out_dir: Path, label_names):
    out_dir.mkdir(parents=True, exist_ok=True)
    name = run["name"]
    eps = np.arange(1, run["epochs"] + 1)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(eps, run["train_loss"], marker="o", markersize=3)
    ax.set_xlabel("epoch"); ax.set_ylabel("train loss")
    ax.set_title(f"{name} — training loss")
    ax.grid(True, alpha=0.3)
    fig.tight_layout(); fig.savefig(out_dir / f"{name}_training_loss.png", dpi=150); plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(eps, run["train_acc"], marker="o", markersize=3, label="train acc")
    ax.plot(eps, run["test_acc"], marker="s", markersize=3, label="test acc")
    ax.set_xlabel("epoch"); ax.set_ylabel("accuracy")
    ax.set_title(f"{name} — accuracy curves")
    ax.legend(); ax.grid(True, alpha=0.3)
    fig.tight_layout(); fig.savefig(out_dir / f"{name}_accuracy_curve.png", dpi=150); plt.close(fig)

    plot_confusion_matrix(
        run["y_test"], run["y_pred"], label_names,
        save_path=out_dir / f"{name}_confusion_matrix.png",
        title=f"{name} — confusion matrix",
    )


## Level 1 ResNet-18 runs

- **L1-fair-50**: same trial-0 50% clip-level labeled subset as the Level 1
  V2 graph SSL.
- **L1-full**: 100% of HF train (no SSL masking).


In [ ]:
L1_DIR = V2_ROOT / "Level1"
L1_FIG = L1_RUN / "figures"
L1_RES = L1_RUN / "results"

spec1 = np.load(PROJECT_ROOT / "Level1" / "Data" / "level1_spectrograms.npz", allow_pickle=True)
feat1 = np.load(PROJECT_ROOT / "Level1" / "Data" / "level1_features.npz", allow_pickle=True)
S1 = spec1["S"].astype(np.float32)
y1 = feat1["labels"].astype(np.int64)
clip_ids1 = feat1["clip_ids"].astype(np.int64)
is_train1 = feat1["is_train"].astype(bool)
label_names1 = list(feat1["label_names"])

mask1 = np.load(V2_ROOT / "Level1" / "Data" / "level1_v2_trial0_mask.npz", allow_pickle=True)
clip_label_mask_l1 = mask1["mask"].astype(bool)

# fair (50% clip-level)
keep_tr_fair = is_train1 & clip_label_mask_l1
keep_te = ~is_train1
print(f"L1-fair-50: train n={keep_tr_fair.sum()}  test n={keep_te.sum()}")
run_l1_fair = train_one(
    "L1-fair-50",
    S1[keep_tr_fair], y1[keep_tr_fair], S1[keep_te], y1[keep_te],
    num_classes=len(label_names1),
)
save_run_artifacts(run_l1_fair, L1_FIG, label_names1)

# full labels
print(f"L1-full: train n={is_train1.sum()}  test n={keep_te.sum()}")
run_l1_full = train_one(
    "L1-full",
    S1[is_train1], y1[is_train1], S1[keep_te], y1[keep_te],
    num_classes=len(label_names1),
)
save_run_artifacts(run_l1_full, L1_FIG, label_names1)


## Level 2 ResNet-18 runs (30%, 50%, 75%, full)


In [ ]:
L2_DIR = V2_ROOT / "Level2"
L2_FIG = L2_RUN / "figures"
L2_RES = L2_RUN / "results"

spec2 = np.load(PROJECT_ROOT / "Level2" / "Data" / "level2_spectrograms.npz", allow_pickle=True)
feat2 = np.load(PROJECT_ROOT / "Level2" / "Data" / "level2_features.npz", allow_pickle=True)
S2 = spec2["S"].astype(np.float32)
y2 = feat2["labels"].astype(np.int64)
clip_ids2 = feat2["clip_ids"].astype(np.int64)
is_train2 = feat2["is_train"].astype(bool)
label_names2 = list(feat2["label_names"])

masks2 = np.load(V2_ROOT / "Level2" / "Data" / "level2_v2_clip_masks.npz", allow_pickle=True)

l2_runs = []
keep_te2 = ~is_train2

for frac_pct in (30, 50, 75):
    cmask = masks2[f"frac_{frac_pct}_trial0"].astype(bool)
    keep_tr = is_train2 & cmask
    name = f"L2-fair-{frac_pct}"
    print(f"{name}: train n={keep_tr.sum()}  test n={keep_te2.sum()}")
    run = train_one(
        name,
        S2[keep_tr], y2[keep_tr], S2[keep_te2], y2[keep_te2],
        num_classes=len(label_names2),
    )
    save_run_artifacts(run, L2_FIG, label_names2)
    l2_runs.append(run)

# full labels
print(f"L2-full: train n={is_train2.sum()}  test n={keep_te2.sum()}")
run_l2_full = train_one(
    "L2-full",
    S2[is_train2], y2[is_train2], S2[keep_te2], y2[keep_te2],
    num_classes=len(label_names2),
)
save_run_artifacts(run_l2_full, L2_FIG, label_names2)
l2_runs.append(run_l2_full)


## Persist a single canonical results file per level


In [ ]:
def serialize_run(r):
    return {
        "name": r["name"],
        "epochs": r["epochs"],
        "n_train": r["n_train"],
        "n_test": r["n_test"],
        "final_acc": r["final_acc"],
        "final_f1": r["final_f1"],
        "wall_seconds": r["wall_seconds"],
        "train_loss": r["train_loss"],
        "train_acc": r["train_acc"],
        "test_acc": r["test_acc"],
    }

l1_results = {
    "device": device,
    "epochs": EPOCHS,
    "lr": LR,
    "batch": BATCH,
    "runs": [serialize_run(run_l1_fair), serialize_run(run_l1_full)],
}
(L1_RES / "resnet_v2_results.json").write_text(json.dumps(l1_results, indent=2), encoding="utf-8")

l2_results = {
    "device": device,
    "epochs": EPOCHS,
    "lr": LR,
    "batch": BATCH,
    "runs": [serialize_run(r) for r in l2_runs],
}
(L2_RES / "resnet_v2_results.json").write_text(json.dumps(l2_results, indent=2), encoding="utf-8")

print("\nLevel 1 results:")
for r in [run_l1_fair, run_l1_full]:
    print(f"  {r['name']:>12s}: n_train={r['n_train']:>4d}  acc={r['final_acc']:.4f}  F1={r['final_f1']:.4f}")
print("\nLevel 2 results:")
for r in l2_runs:
    print(f"  {r['name']:>12s}: n_train={r['n_train']:>4d}  acc={r['final_acc']:.4f}  F1={r['final_f1']:.4f}")


## Combined accuracy-vs-labels plot for Level 2 (Fix 1 finishing touch)

X-axis = label fraction (clip-level). Lines:
  - Graph SSL (label propagation, default k=10, σ=median) from Fix 1.
  - ResNet-18 (fair, 50 epochs) — same clip-level subsets.
  - ResNet-18 (100% labels, 50 epochs) — horizontal reference.


In [ ]:
from v2.utils.runs import find_latest_result
gss_path = find_latest_result(L2_DIR, "level2_results_v2_fix1.json")
if gss_path is None:
    raise FileNotFoundError(
        "No Level 2 Fix 1 results found yet. Run "
        "v2/Level2/level2_classification_v2.ipynb first."
    )
print(f"reading graph SSL results from: {gss_path}")
gss = json.loads(gss_path.read_text(encoding="utf-8"))

fracs = [0.30, 0.50, 0.75]
gss_acc = [gss["v2_clip_split"][f"{int(f*100)}pct"]["acc_mean"] for f in fracs]
gss_std = [gss["v2_clip_split"][f"{int(f*100)}pct"]["acc_std"] for f in fracs]

cnn_fair_acc = [run["final_acc"] for run in l2_runs[:3]]  # 30, 50, 75
cnn_full_acc = run_l2_full["final_acc"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar([100*f for f in fracs], gss_acc, yerr=gss_std, marker="o",
            capsize=4, label="Graph SSL (label prop, k=10, σ=median)")
ax.plot([100*f for f in fracs], cnn_fair_acc, marker="s",
        label="ResNet-18 (50 epochs, same clip-level labels)")
ax.axhline(cnn_full_acc, color="tab:red", linestyle="--",
           label=f"ResNet-18 (100% train, 50 epochs) = {cnn_full_acc:.3f}")
ax.set_xlabel("label fraction (clip-level)")
ax.set_ylabel("test accuracy")
ax.set_title("Level 2 V2 — accuracy vs label fraction")
ax.set_xticks([30, 50, 75])
ax.grid(True, alpha=0.3); ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
fig.savefig(L2_FIG / "accuracy_vs_labels.png", dpi=150)
plt.close(fig)
print("saved", L2_FIG / "accuracy_vs_labels.png")
